In [ ]:
pip install numpy pandas matplotlib scikit-learn scipy shap joblib openpyxl

In [ ]:
# ======================================================================
# LightGBM - High-Capacity Tuning for Max R²
# ======================================================================
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from lightgbm import LGBMRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Input Path from your setup
ml_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\XGboost_Output"
input_csv = os.path.join(ml_dir, "lagged_ml_ready.csv")

# Data Loading
df = pd.read_csv(input_csv)
drop_columns = ["VCI", "Row_Idx", "Col_Idx", "Year"]
X = df.drop(columns=drop_columns)
y = df["VCI"].copy()

target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, shuffle=True
)

# ======================================================================
# HIGH-PERFORMANCE / ENHANCED CAPACITY LIGHTGBM PARAMETERS
# ======================================================================
print("\nTraining High-Capacity LightGBM Model...")
lgbm_model = LGBMRegressor(
    n_estimators=400,          # Increased trees to capture complex lag relationships
    learning_rate=0.05,        # Smaller step size for better convergence with high tree count
    max_depth=10,               # Deeper tree hierarchy allowed
    num_leaves=128,             # Significantly increased complexity (main driver for LightGBM)
    min_child_samples=10,       # Lower value allows leaves to isolate finer local patterns
    subsample=0.85,             # Row subsampling
    subsample_freq=1,           
    colsample_bytree=0.85,      # Feature subsampling
    reg_alpha=0.05,             # Minimal L1 regularization
    reg_lambda=0.5,             # Mild L2 regularization
    random_state=RANDOM_STATE,
    verbosity=-1
)

lgbm_model.fit(X_train, y_train)
print("Training Completed.")

# Performance Predictions
train_pred = lgbm_model.predict(X_train)
test_pred = lgbm_model.predict(X_test)

# Metrics Calculation
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print("\n" + "="*75)
print("LIGHTGBM HIGH-PERFORMANCE METRICS")
print("="*75)
print(f"Train R²  : {train_r2:.4f}")
print(f"Test R²   : {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Train MAE : {train_mae:.4f}")
print(f"Test MAE  : {test_mae:.4f}")
print("="*75)

# Journal Style Visualization
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["figure.dpi"] = 300
fig = plt.figure(figsize=(12, 5), facecolor='white')
box_props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='lightgray')

# Training Plot
ax1 = plt.subplot(1, 2, 1, facecolor='white')
plt.scatter(y_train, train_pred, alpha=0.15, color='tab:cyan', s=1)
plt.plot([0, 1], [0, 1], 'k--', lw=1.5)
train_text = f"R²: {train_r2:.4f}\nRMSE: {train_rmse:.4f}\nMAE: {train_mae:.4f}"
ax1.text(0.04, 0.96, train_text, transform=ax1.transAxes, fontsize=10, bbox=box_props, verticalalignment='top', horizontalalignment='left')
plt.xlim([-0.05, 1.05]); plt.ylim([-0.05, 1.05])
plt.xlabel('Actual VCI')
plt.ylabel('Predicted VCI')
plt.title('Training: Actual vs Predicted')
plt.grid(True, linestyle=':', alpha=0.4)

# Testing Plot
ax2 = plt.subplot(1, 2, 2, facecolor='white')
plt.scatter(y_test, test_pred, alpha=0.2, color='tab:cyan', s=1)
plt.plot([0, 1], [0, 1], 'k--', lw=1.5)
test_text = f"R²: {test_r2:.4f}\nRMSE: {test_rmse:.4f}\nMAE: {test_mae:.4f}"
ax2.text(0.04, 0.96, test_text, transform=ax2.transAxes, fontsize=10, bbox=box_props, verticalalignment='top', horizontalalignment='left')
plt.xlim([-0.05, 1.05]); plt.ylim([-0.05, 1.05])
plt.xlabel('Actual VCI')
plt.ylabel('Predicted VCI')
plt.title('Test: Actual vs Predicted')
plt.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()